<a href="https://colab.research.google.com/github/tipismaca/Trabajo-Bebidas-del-Maipo-SA/blob/main/CasodeEstudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf Trabajo-Bebidas-del-Maipo-SA #lo que hace esta linea es que elimina la clonacion anterior (la con coma)
!git clone https://github.com/tipismaca/Trabajo-Bebidas-del-Maipo-SA.git

Cloning into 'Trabajo-Bebidas-del-Maipo-SA'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 85 (delta 33), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 56.41 KiB | 1.41 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [ ]:
import pandas as pd
import numpy as np
import json


In [ ]:
# 1. Definir la ruta base de la carpeta clonada
ruta_base = 'Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/'

# 2. Leer los archivos CSV sin modificar los datos crudos originales
df_calidad = pd.read_csv(ruta_base + 'calidad_llenado.csv')
df_especificaciones = pd.read_csv(ruta_base + 'especificaciones_calidad.csv')

# 3. Agrupar por línea y formato como exige la Parte A
# Se calcula la media y el rango (máximo - mínimo) para cada subgrupo (n=5)
resumen_subgrupos = df_calidad.groupby(['linea', 'formato_L', 'subgrupo']).agg(
    media_volumen=('volumen_mL', 'mean'),
    rango_volumen=('volumen_mL', lambda x: np.ptp(x)) # np.ptp (peak-to-peak) calcula max - min
).reset_index()

# Mostrar las primeras filas para verificar que cargó y calculó bien
display(resumen_subgrupos.head())
display(df_especificaciones)

In [ ]:
# 1. Definir la ruta base
ruta_base = 'Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/'

# 2. Cargar archivos
df_llenado = pd.read_csv(ruta_base + 'calidad_llenado.csv')
df_especs = pd.read_csv(ruta_base + 'especificaciones_calidad.csv')
df_atributos = pd.read_csv(ruta_base + 'calidad_atributos.csv')

# 3. Agrupar subgrupos (n = 5)
subgrupos = df_llenado.groupby(['linea', 'formato_L', 'subgrupo']).agg(
    X_bar=('volumen_mL', 'mean'),
    R=('volumen_mL', lambda x: x.max() - x.min()),
    semana=('semana_registro', 'first'),
    turno=('turno', 'first')
).reset_index()

# Constantes para n = 5
A2, D3, D4, d2 = 0.577, 0.0, 2.114, 2.326

# 4. Fase I (Semanas 1 a 6)
fase1 = subgrupos[subgrupos['semana'] <= 6]

print("=== LÍMITES DE CONTROL FASE I (SEMANAS 1 A 6) ===")
limites_f1 = {}
for (linea, formato), g in fase1.groupby(['linea', 'formato_L']):
    X_db = g['X_bar'].mean()
    R_b = g['R'].mean()
    UCL_X, LCL_X = X_db + A2 * R_b, X_db - A2 * R_b
    limites_f1[(linea, formato)] = {'X_db': X_db, 'R_b': R_b, 'UCL_X': UCL_X, 'LCL_X': LCL_X, 'sigma_corto': R_b / d2}

    print(f"📍 {linea} ({formato}L) -> LCL: {LCL_X:.2f} mL | Media: {X_db:.2f} mL | UCL: {UCL_X:.2f} mL")

# 5. Gráfico de Carta X-bar para Línea L1 (0.5 L)
l1_data = subgrupos[(subgrupos['linea'] == 'L1') & (subgrupos['formato_L'] == 0.5)].sort_values('subgrupo')
lim_l1 = limites_f1[('L1', 0.5)]

plt.figure(figsize=(10, 4))
plt.plot(l1_data['subgrupo'], l1_data['X_bar'], marker='o', linestyle='-', color='b', markersize=3, label='Media Subgrupo (X-bar)')
plt.axhline(lim_l1['X_db'], color='green', linestyle='--', label=f"Media Fase I ({lim_l1['X_db']:.2f} mL)")
plt.axhline(lim_l1['UCL_X'], color='red', linestyle='--', label=f"UCL ({lim_l1['UCL_X']:.2f} mL)")
plt.axhline(lim_l1['LCL_X'], color='red', linestyle='--', label=f"LCL ({lim_l1['LCL_X']:.2f} mL)")
plt.axvline(x=144, color='gray', linestyle=':', label='Fin Fase I (Semana 6)')

plt.title('Carta de Control X-bar: Línea L1 (0.5 L)')
plt.xlabel('Subgrupo')
plt.ylabel('Volumen Promedio (mL)')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()